# Project setup

**Goal of this phase:** prove the workshop works before we build anything in it.

By the end of this notebook you will have checked that:
1. the `asthunt` environment has every library we need,
2. PyTorch can use your M3 Pro GPU (Apple calls it **MPS**),
3. our own package `asthunt` imports, and its config loader gives absolute paths,
4. the raw ZTF images are where the config says they are, and we can open one.

Read `docs/project_setup.md` alongside this notebook. It explains the *why*.

## 1. Library versions
Write these down. If a result changes months from now, versions are the first suspect.

In [ ]:
import sys
import numpy, pandas, astropy, sep, galsim, sklearn, xgboost, torch, matplotlib

print("python ", sys.version.split()[0])
for lib in [numpy, pandas, astropy, sep, galsim, sklearn, xgboost, torch, matplotlib]:
    print(f"{lib.__name__:11s}", lib.__version__)

## 2. Is the GPU available?
`mps` = Apple-silicon GPU. We will train the CNN on it later. If this says `cpu`, everything still works, only slower.

In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("training device:", device)

x = torch.randn(1000, 1000, device=device)
print("matrix multiply on", (x @ x).device, "works")

## 3. Our own package and its config
No more hard-coded `/Users/...` paths: every stage asks `load_config` where things live.

In [ ]:
from asthunt.config.load_config import PROJECT_ROOT, load_config

config = load_config("project")
print("project root:", PROJECT_ROOT)
print("seed:", config["seed"])
for name, path in config["paths"].items():
    print(f"{name:10s} {path}  exists={path.exists()}")

## 4. Find the raw images

In [ ]:
fits_files = sorted(config["paths"]["raw"].glob("ztf_test/sci/**/*sciimg.fits"))
for f in fits_files:
    print(f.name)
print(len(fits_files), "science images")

## 5. Open one image
A FITS file = a **header** (metadata: time, zeropoint, seeing) + a **data** array (pixel values).
We show the image with a percentile stretch, because raw values span a huge range and a linear stretch shows almost nothing.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

with fits.open(fits_files[0]) as hdul:
    header = hdul[0].header
    image = hdul[0].data.astype(float)

for key in ["OBSJD", "FIELDID", "CCDID", "QID", "FILTER", "MAGZP", "SEEING", "PIXSCALE"]:
    print(f"{key:9s}", header.get(key))
print("image shape:", image.shape)

vmin, vmax = np.nanpercentile(image, [5, 99])
plt.figure(figsize=(7, 7))
plt.imshow(image, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
plt.title(fits_files[0].name, fontsize=8)
plt.colorbar(label="counts (DN)")
plt.show()

## Exercises
1. Change the stretch to `[0, 100]` percentiles. Why does the image turn almost black?
2. Print `header["OBSJD"]` for all images. How many days apart are they? Why is that a problem for linking asteroids? (Hint: problem 2 in `PLAN.md`.)
3. Set `device = "cpu"` and time the matrix multiply with `%timeit`. Then do the same on `mps`.